In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv(
    "../data/processed/elo_features_with_league.csv",
    parse_dates=["date"]
)


In [4]:


merged_df = pd.read_csv("../data/processed/elo_features_with_league.csv")
merged_df["date"] = pd.to_datetime(df["date"])
worlds_2026 = merged_df[
    (merged_df["league"] == "WLDs") &
    (merged_df["date"].dt.year == 2026)
]

print("Matches:", len(worlds_2026))
print("First match:", worlds_2026["date"].min())
print("Last match:", worlds_2026["date"].max())
print("Teams:", worlds_2026["team_a"].nunique())

Matches: 63
First match: 2026-09-17 09:11:42+00:00
Last match: 2026-10-26 21:54:37+00:00
Teams: 14


In [9]:
ratings_a = worlds_2026[
    ["date", "team_a", "post_elo_a"]
].rename(columns={
    "team_a": "team",
    "post_elo_a": "elo"
})

ratings_b = worlds_2026[
    ["date", "team_b", "post_elo_b"]
].rename(columns={
    "team_b": "team",
    "post_elo_b": "elo"
})

all_ratings = pd.concat(
    [ratings_a, ratings_b],
    ignore_index=True
)

In [10]:
worlds_2026 = worlds_2026.copy()

worlds_2026["expected_a"] = (
    1 / (1 + 10 ** (-worlds_2026["elo_diff"] / 400))
)

In [11]:

K = 24
# Calculate post-match Elo for Team A
worlds_2026["post_elo_a"] = (
    worlds_2026["pre_elo_a"]
    + K * (worlds_2026["result_a"] - worlds_2026["expected_a"])
)

# Calculate post-match Elo for Team B
worlds_2026["post_elo_b"] = (
    worlds_2026["pre_elo_b"]
    + K * (
        (1 - worlds_2026["result_a"])
        - (1 - worlds_2026["expected_a"])
    )
)


In [12]:
# Display results
print(
    worlds_2026[
        [
            "date",
            "team_a",
            "team_b",
            "pre_elo_a",
            "pre_elo_b",
            "post_elo_a",
            "post_elo_b",
        ]
    ].head()
)

                           date           team_a       team_b    pre_elo_a  \
95085 2026-09-17 09:11:42+00:00  Invictus Gaming  Top Esports  1766.305531   
95086 2026-09-17 10:14:41+00:00  Invictus Gaming  Top Esports  1777.067599   
95087 2026-09-17 11:11:25+00:00  Invictus Gaming  Top Esports  1763.099758   
95088 2026-09-17 12:10:40+00:00  Invictus Gaming  Top Esports  1750.081309   
95105 2026-09-18 09:12:27+00:00        JD Gaming      Team WE  1709.536493   

         pre_elo_b   post_elo_a   post_elo_b  
95085  1730.335741  1777.067599  1719.573674  
95086  1719.573674  1763.099758  1733.541514  
95087  1733.541514  1750.081309  1746.559963  
95088  1746.559963  1737.959690  1758.681582  
95105  1710.684431  1697.576141  1722.644783  


In [13]:
latest_ratings = (
    all_ratings
    .sort_values("date")
    .drop_duplicates(subset="team", keep="last")
    .sort_values("elo", ascending=False)
    .reset_index(drop=True)
)

latest_ratings["rank"] = latest_ratings.index + 1

print(latest_ratings[["rank", "team", "elo", "date"]].to_string(index=False))

 rank                team         elo                      date
    1               Gen.G 1974.697685 2026-10-25 22:43:24+00:00
    2     Bilibili Gaming 1908.463114 2026-10-26 18:03:09+00:00
    3          G2 Esports 1870.203091 2026-10-25 18:11:29+00:00
    4           Dplus Kia 1867.970444 2026-10-25 20:10:09+00:00
    5 Hanwha Life Esports 1867.730515 2026-10-25 17:19:41+00:00
    6                  T1 1839.362372 2026-10-26 17:12:02+00:00
    7        Karmine Corp 1838.488298 2026-10-26 18:03:09+00:00
    8                 LØS 1832.870123 2026-10-26 20:01:50+00:00
    9  Team Secret Whales 1826.312222 2026-10-26 20:01:50+00:00
   10         Team Liquid 1823.581101 2026-10-26 21:54:37+00:00
   11                LYON 1816.891363 2026-10-25 22:43:24+00:00
   12     Invictus Gaming 1784.344594 2026-10-25 17:19:41+00:00
   13         Top Esports 1781.102839 2026-10-25 18:11:29+00:00
   14     Anyone's Legend 1778.186321 2026-10-26 21:54:37+00:00
   15        Movistar KOI 1749.018377 20